# Death Knight Lab

Random beginnings, learned strategies.

**Experiment 0: create an untrained deck population.** This notebook does not simulate games or train an agent yet. It uses a frozen Core-only card pool, not all Standard cards. Current live legality still needs verification.

In [ ]:
from pathlib import Path
import sys, json
from collections import Counter
root = Path.cwd()
if not (root / "lab.py").exists():
    root = root.parent
sys.path.insert(0, str(root))
from lab import load_cards, population, describe, validate
cards = load_cards()
print(f"{len(cards)} cards in our initial pool")

## Generate the starting population

Ten decks for each of the ten three-rune combinations. No handpicked strong cards, mana curve targets, match data, or pretrained weights. Sampling card copies introduces a known prior: non-legendary cards have two available slots; legendary cards have one.

In [ ]:
SEED = 42
DECKS_PER_RUNE_PROFILE = 10
decks = population(cards, per_profile=DECKS_PER_RUNE_PROFILE, seed=SEED)
print(f"Generated {len(decks)} decks; training games played: 0")
assert all(not validate(d["cards"], cards, tuple(d["runes"].values())) for d in decks)

In [ ]:
index = 0  # Change this to inspect another deck.
print(decks[index]["runes"])
print(describe(decks[index]["cards"], cards))

## Inspect the starting mana curve

This is a description of the random population, not a measure of strength.

In [ ]:
lookup = {c["id"]: c for c in cards}
curve = Counter(min(lookup[i]["cost"], 10) for d in decks for i in d["cards"])
for mana in range(11):
    mean = curve[mana] / len(decks)
    print(f'{str(mana) + ("+" if mana == 10 else ""):>3}: {mean:4.1f} {"█" * round(mean * 5)}')

In [ ]:
output = root / "runs"
output.mkdir(exist_ok=True)
(output / "generation_000.json").write_text(json.dumps({"seed": SEED, "status": "untrained", "decks": decks}, indent=2))
print("Saved generation_000.json")

## Next: make the games trustworthy

Before learning, implement or integrate a verified simulator for this pool: rune constraints, Corpses, Ghoul Charge, combat, card effects, random generation, and hidden information. Every generated card also needs support. Unsupported effects must stop a game, never silently become vanilla effects.

Then train a shared policy through self-play, freeze it for deck evaluation, alternate starting player, compare against earlier policies, and evolve card choices. Mirror matches measure success against this population, not the wider Standard field. Keep held-out opponents and evaluate new mutations on fresh games.

No fitness scores or win-rate charts are shown because no games have been played.